In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression,Ridge,SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn import tree


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn import svm

import matplotlib.dates as mdates
import matplotlib.pyplot as plt



In [ ]:
# Agent Guideline (You are eligible to modify entire notebook but pls follow instruction)

# I want make a function that do time plot of hourly classic bike rental prediction (blue) vs true (orange)

# It must handle the time span of 1 jan to dec 31

# After that i want you to integrate with other model experiment, right now we only see sample right?
# I wanna see graph isntead of those, so you have to repalce that with our plot method

import re
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt


def model_report(
    test_df,
    y_true,
    y_pred,
    target_scaler,
    year,
    model_name,
    mse,
):
    """Report full-year hourly predictions, true rentals, and model MSE."""
    time_columns = ["Month", "Day", "Hour"]
    missing_columns = [column for column in time_columns if column not in test_df.columns]
    if missing_columns:
        raise ValueError(f"test_df is missing time columns: {missing_columns}")

    # The y arrays share the same row order as test_df, so only their values are needed.
    true_scaled = np.asarray(y_true).reshape(-1, 1)
    pred_scaled = np.asarray(y_pred).reshape(-1, 1)
    if len(true_scaled) != len(test_df) or len(pred_scaled) != len(test_df):
        raise ValueError("test_df, y_true, and y_pred must contain the same number of rows")
    true_rentals = target_scaler.inverse_transform(true_scaled).ravel()
    predicted_rentals = target_scaler.inverse_transform(pred_scaled).ravel()

    # Build real calendar timestamps from test_df and sort them across the complete year.
    calendar_parts = test_df[time_columns].copy().rename(
        columns={"Month": "month", "Day": "day", "Hour": "hour"}
    )
    calendar_parts["year"] = int(year)
    prediction_data = pd.DataFrame(
        {
            "Date": pd.to_datetime(calendar_parts[["year", "month", "day", "hour"]]),
            "Prediction": predicted_rentals,
            "True": true_rentals,
        }
    ).sort_values("Date")

    # Save four vertically stacked three-month phases across January 1 through December 31.
    phase_ranges = [(1, 3), (4, 6), (7, 9), (10, 12)]
    plot_output_dir = Path("output/plot_artifacts")
    plot_output_dir.mkdir(parents=True, exist_ok=True)
    figure, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(16, 12),
        sharey=False,
    )
    for axis, (start_month, end_month) in zip(axes, phase_ranges):
        phase_data = prediction_data[
            prediction_data["Date"].dt.month.between(start_month, end_month)
        ]
        axis.plot(
            phase_data["Date"],
            phase_data["Prediction"],
            color="blue",
            linewidth=0.8,
            label="Prediction",
        )
        axis.plot(
            phase_data["Date"],
            phase_data["True"],
            color="#d97706",  # Dark orange remains visible against the prediction line.
            linewidth=0.9,
            label="True",
        )
        phase_start = pd.Timestamp(int(year), start_month, 1)
        phase_end = pd.Timestamp(int(year), end_month, 1) + pd.offsets.MonthBegin(1)
        axis.set_xlim(phase_start, phase_end)
        axis.set_ylim(0, 2000)  # Cap rental units at 2,000 for consistent plot height.
        axis.xaxis.set_major_locator(mdates.MonthLocator())
        axis.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        axis.set_title(f"Months {start_month}–{end_month}")
        axis.set_xlabel("Date")
        axis.set_ylabel("Classic Bike Rentals")

    # Keep one shared legend so the four panels remain uncluttered.
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.99, 0.99))
    figure.suptitle(
        f"{model_name}: Hourly Classic Bike Rentals ({year}, MSE: {mse:.4f}, Y-axis capped at 2,000)",
        y=0.995,
    )
    figure.tight_layout(rect=[0, 0, 0.94, 0.97])
    safe_model_name = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    plot_path = plot_output_dir / f"{safe_model_name}_{int(year)}.png"
    figure.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(figure)  # Keep the notebook output free of inline figures.


### randomness value

In [ ]:
sklearn_random_state = 42

### Load data

In [ ]:
s3_data_path = "s3://mlops-project-bucket-602343785232-ap-southeast-1-an/extract_data/bike_rental_y_2022_2025_m_1_12.csv"
df = pd.read_csv(s3_data_path)


In [ ]:
df.columns

In [ ]:
df['total_classic_bike_rental'] = df['classic_bike_casual_count'] + df['classic_bike_member_count']

### Preprocessing

#### - select hourly range of 6 to 19

In [ ]:
# AGENT TASK
# Select the requested hourly range and model columns in a fixed order.
processed_df = (
    df.loc[
        df["Hour"].between(6, 19),  # Include hours 6 through 19.
        [
            "Year",
            "Month",
            "Day",
            "Hour",
            "total_classic_bike_rental",
            "temperature",
        ],
    ]
    .sort_values(["Year", "Month", "Day", "Hour"])
    .reset_index(drop=True)
)
display(processed_df.head())
print("processed_df shape:", processed_df.shape)


### Train Test Partitioning

#### - select 2022 - 2024 as training and 2025 as testing

In [ ]:
# AGENT TASK
# Use 2022–2024 for training and hold out 2025 for testing.
train_df = processed_df.loc[
    processed_df["Year"].between(2022, 2024)
].drop(columns=["Year"]).copy()
test_df = processed_df.loc[
    processed_df["Year"] == 2025
].drop(columns=["Year"]).copy()

# Fit scalers only on training data, then reuse them for the test data.
classic_bike_rental_scaler = StandardScaler()
temperature_scaler = StandardScaler()
train_df["scaled_total_classic_bike_rental"] = classic_bike_rental_scaler.fit_transform(
    train_df[["total_classic_bike_rental"]]
).ravel()
test_df["scaled_total_classic_bike_rental"] = classic_bike_rental_scaler.transform(
    test_df[["total_classic_bike_rental"]]
).ravel()
train_df["scaled_temperature"] = temperature_scaler.fit_transform(
    train_df[["temperature"]]
).ravel()
test_df["scaled_temperature"] = temperature_scaler.transform(
    test_df[["temperature"]]
).ravel()

# Use one numeric dtype for all model data while preserving dataframe column names.
train_df = train_df.astype(np.float64)
test_df = test_df.astype(np.float64)

# Confirm the split, retained source columns, scaled columns, and removal of Year.
print("Training source years: 2022–2024")
print("Testing source year: 2025")
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("Year in train_df:", "Year" in train_df.columns)
print("Year in test_df:", "Year" in test_df.columns)
print("train_df dtypes:", train_df.dtypes.unique())
print("test_df dtypes:", test_df.dtypes.unique())
print(
    "Scaled columns present:",
    all(
        column in train_df.columns
        for column in ["scaled_total_classic_bike_rental", "scaled_temperature"]
    ),
)


In [ ]:
train_df.columns

In [ ]:
features = ['Month', 'Day', 'Hour', 'scaled_temperature']
target = ['scaled_total_classic_bike_rental']
X_train,Y_train = train_df[features],train_df[target]
X_test,Y_test = test_df[features],test_df[target]

### Linear Regression

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year Linear Regression results.
LinearRegressionModel = LinearRegression()
LinearRegressionModel.fit(X_train, Y_train)
Y_pred = LinearRegressionModel.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
train_score = LinearRegressionModel.score(X_train, Y_train)
print(f"Linear Regression MSE error: {error}")
print(f"Linear Regression training R²: {train_score}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="Linear Regression",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


### Ridge Regression

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year Ridge Regression results.
RidgeRegressionModel = Ridge(random_state=sklearn_random_state)
RidgeRegressionModel.fit(X_train, Y_train)
Y_pred = RidgeRegressionModel.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"Ridge Regression MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="Ridge Regression",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


### SVM

#### SVM - rbf

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year SVM RBF results.
SVM_Model = svm.SVR(degree=4)
SVM_Model.fit(X_train, Y_train)
Y_pred = SVM_Model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"SVM RBF MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="SVM RBF",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


#### SVM - linear

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year linear-kernel SVM results.
SVM_Model = svm.SVR(kernel="linear", degree=4)
SVM_Model.fit(X_train, Y_train)
Y_pred = SVM_Model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"SVM Linear MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="SVM Linear",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


#### SVM - poly

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year polynomial-kernel SVM results.
SVM_Model = svm.SVR(kernel="poly", degree=4)
SVM_Model.fit(X_train, Y_train)
Y_pred = SVM_Model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"SVM Polynomial MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="SVM Polynomial",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


#### SVM - sigmoid

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year sigmoid-kernel SVM results.
SVM_Model = svm.SVR(kernel="sigmoid", degree=4)
SVM_Model.fit(X_train, Y_train)
Y_pred = SVM_Model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"SVM Sigmoid MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="SVM Sigmoid",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


### SDG regressor

In [ ]:
# MAIN CELL
# Train, evaluate, and report the full-year SGD Regressor results.
SGD_model = SGDRegressor(
    max_iter=10000,
    early_stopping=True,
    random_state=sklearn_random_state,
    eta0=1e-5,
    learning_rate="constant",
)
SGD_model.fit(
    X_train.to_numpy(dtype=np.float64),
    Y_train.to_numpy(dtype=np.float64).ravel(),
)
Y_pred = SGD_model.predict(X_test.to_numpy(dtype=np.float64))
error = mean_squared_error(Y_test, Y_pred)
print(f"SGD Regressor MSE error: {error}")
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="SGD Regressor",
)


In [ ]:
# OUT OF PLACE
# Evaluation/plot logic moved into the corresponding MAIN CELL.


### KNeighborsRegressor

#### KNN weight = uniform

In [ ]:
# MAIN CELL
# Train KNN with uniform neighbor weights.
KNR_model = KNeighborsRegressor(weights="uniform")
KNR_model.fit(X_train, Y_train)

# Predict and report test error.
Y_pred = KNR_model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"KNN (uniform) MSE error {error}")

# Plot predictions across the full 2025 test year.
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="KNN (Uniform)",
)


#### KNN weight = distance

In [ ]:
# MAIN CELL
# Train KNN with distance-based neighbor weights.
KNR_model = KNeighborsRegressor(weights="distance")
KNR_model.fit(X_train, Y_train)

# Predict and report test error.
Y_pred = KNR_model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"KNN (distance) MSE error {error}")

# Plot predictions across the full 2025 test year.
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="KNN (Distance)",
)


### DecisionTreeRegressor

In [ ]:
# MAIN CELL
# Train the decision-tree regressor.
DTR_model = tree.DecisionTreeRegressor(
    random_state=sklearn_random_state,
    max_depth=50,
    min_samples_split=200,
    min_samples_leaf=200,
    criterion="squared_error",
)
DTR_model.fit(X_train, Y_train)

# Predict and report test error.
Y_pred = DTR_model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"Decision Tree Regressor MSE error {error}")

# Plot predictions across the full 2025 test year.
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="Decision Tree Regressor",
)


### MLPRegressor

In [ ]:
# MAIN CELL
# Train the multilayer perceptron regressor.
MLP_model = MLPRegressor(
    random_state=sklearn_random_state,
    learning_rate_init=1e-5,
    early_stopping=True,
    max_iter=1000,
)
MLP_model.fit(X_train, Y_train)

# Predict and report test error.
Y_pred = MLP_model.predict(X_test)
error = mean_squared_error(Y_test, Y_pred)
print(f"MLP Regressor MSE error {error}")

# Plot predictions across the full 2025 test year.
model_report(
    test_df,
    Y_test,
    Y_pred,
    classic_bike_rental_scaler,
    mse=error,
    year=2025,
    model_name="MLP Regressor",
)
